# Bandit contextual causal — qual assunto mandar, ou não mandar nada

Protótipo de um motor que decide, para cada cliente novo, **qual assunto** enviar
nos primeiros dias de conta — ou **não enviar nada**, se o silêncio for melhor.

A diferença para um bandit contextual comum é o que se otimiza. Um bandit clássico
maximiza a conversão observada e aprende a mandar mensagem para quem já ia converter
sozinho. Aqui cada braço estima o **uplift incremental (CATE)** contra um grupo de
controle, pelo método do *transformed outcome* (Athey & Imbens):

$$Y^* = Y \cdot \left(\frac{W}{p} - \frac{1-W}{1-p}\right) = Y \cdot \frac{W - p}{p(1-p)},
\qquad \mathbb{E}[Y^* \mid X] = \text{CATE}(X)$$

O controle não é só uma linha de base estatística: é uma **ação de primeira classe**.
Se nenhum assunto tiver uplift positivo para aquele cliente, a decisão ótima é calar.

## Como este notebook está organizado

1. A classe `CausalContextualBandit` — decisão, logging de propensões e treino.
2. Um **simulador com CATE verdadeiro conhecido**. Sem ele não dá para afirmar que a
   implementação está certa: num log real o uplift nunca é observado.
3. Três validações: o modelo recupera o CATE? a política decide bem? o ciclo fechado
   converge quando o motor passa a gerar o próprio log?
4. O caminho de produção, um cliente por vez.

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

## 1. O motor

Quatro decisões de design que valem o comentário:

- **Contrato de features explícito.** O modelo recebe `feature_names` e só enxerga
  essas colunas, nessa ordem, no treino e no scoring. Sem isso, uma coluna extra no
  log (`cliente_id`, `safra`) ou uma reordenação viraria feature silenciosamente —
  o modelo treinaria errado sem levantar erro nenhum.
- **Warm-up randomiza uniforme.** Assumir uplift zero para um modelo não treinado
  não força exploração: força o *controle*, porque a regra gulosa manda calar quando
  nada é positivo. O motor concentraria ~84% do tráfego no silêncio e nunca aprenderia.
- **Piso de tráfego no controle.** O controle é o contrafactual dos quatro modelos
  ao mesmo tempo. Quando ele deixa de ser a ação gulosa, o ε-greedy o reduz a ~4%
  do tráfego e os quatro modelos degradam juntos. A célula de validação 3 mostra o
  tamanho do efeito.
- **Propensão logada por braço.** Para estimar o uplift do `pix` é preciso saber qual
  era a chance do `pix` **até para quem caiu no controle**. Logar só a propensão da
  ação escolhida inviabiliza o treino.

In [ ]:
class CausalContextualBandit:
    """
    Bandit contextual causal: escolhe QUAL assunto mandar (ou não mandar nada)
    maximizando o UPLIFT incremental de conversão, não a conversão bruta.

    Cada tratamento tem um modelo que estima o CATE contra o controle, treinado
    pelo método do "transformed outcome" (Athey & Imbens):

        Y* = Y * (W - p) / (p * (1 - p))    com    E[Y* | X] = CATE(X)

    O braço 'controle' (não mandar nada) é uma ação de primeira classe: se nenhum
    assunto tiver uplift positivo, o silêncio é a decisão ótima.
    """

    # Y* é um alvo de variância brutal (pesos IPW multiplicam um 0/1 por até 1/p).
    # Sem regularização pesada o XGBoost decora ruído e inventa uplift onde não há.
    # min_child_weight está em nº de linhas por folha: ajuste junto com o volume.
    # Cuidado ao lê-lo como "sinal por folha": ele conta LINHAS, e as linhas com
    # Y=0 entram em Y* como zero exato. O que sustenta uma folha é o convertido.
    DEFAULT_MODEL_PARAMS = dict(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=200,
        reg_lambda=5.0,
    )

    def __init__(
        self,
        treatments,
        feature_names,
        epsilon=0.15,
        control_floor=0.15,
        control_name="controle",
        min_samples=200,
        min_group_samples=100,
        min_conversions=30,
        propensity_floor=0.02,
        uplift_margin=0.0,
        max_batches_stale=3,
        model_params=None,
        random_state=None,
    ):
        """
        treatments        : assuntos possíveis (ex: ['pix', 'pagamento', ...]).
        feature_names      : CONTRATO de features. A ordem aqui é a ordem que o
                             modelo enxerga, tanto no treino quanto no scoring.
        epsilon            : fração do tráfego reservada à exploração uniforme.
        control_floor      : fração do tráfego GARANTIDA ao controle, além do que
                             a exploração já dá. O controle é o contrafactual de
                             TODOS os modelos de uplift: se ele secar quando deixa
                             de ser a ação gulosa, os quatro modelos degradam
                             juntos. Custa conversão hoje, compra precisão amanhã.
        min_samples        : mínimo de linhas (tratamento + controle) para treinar.
        min_group_samples  : mínimo de linhas em CADA grupo, tratado e controle.
                             O total não identifica nada por si: 600 linhas com 3
                             tratadas produzem um CATE com o sinal a esmo.
        min_conversions    : mínimo de convertidos no subset. Quem carrega sinal
                             em Y* é o convertido, não a linha — ver a nota nas
                             guardas de identificação em train_batch.
        propensity_floor   : clipa p em [floor, 1-floor]. Limita o peso IPW a
                             1/floor, trocando um pouco de viés por variância.
        uplift_margin      : só manda mensagem se o uplift estimado passar disso.
                             > 0 protege contra o winner's curse (ver nota abaixo).
        max_batches_stale  : após N batches sem conseguir retreinar um braço, o
                             modelo dele é considerado obsoleto e volta ao warm-up.
        """
        if not treatments:
            raise ValueError("É preciso ao menos um tratamento.")
        if control_name in treatments:
            raise ValueError(f"'{control_name}' é o braço de controle, não um tratamento.")
        if epsilon + control_floor > 1:
            raise ValueError("epsilon + control_floor não pode passar de 1.")

        self.treatments = list(treatments)
        self.feature_names = list(feature_names)
        self.control_name = control_name
        self.epsilon = epsilon
        self.control_floor = control_floor
        self.min_samples = min_samples
        self.min_group_samples = min_group_samples
        self.min_conversions = min_conversions
        self.propensity_floor = propensity_floor
        self.uplift_margin = uplift_margin
        self.max_batches_stale = max_batches_stale
        self.rng = np.random.default_rng(random_state)

        params = {**self.DEFAULT_MODEL_PARAMS, **(model_params or {})}
        self.models = {trt: XGBRegressor(**params) for trt in self.treatments}

        self.is_trained = {trt: False for trt in self.treatments}
        self.batches_since_fit = {trt: 0 for trt in self.treatments}

    # ------------------------------------------------------------------ #
    # Definição dos braços e das colunas de log
    # ------------------------------------------------------------------ #
    @property
    def arms(self):
        return self.treatments + [self.control_name]

    @property
    def prop_cols(self):
        return [f"p_{arm}" for arm in self.arms]

    @property
    def in_warmup(self):
        """Enquanto QUALQUER braço não tem modelo válido, randomizamos uniforme."""
        return not all(self.is_trained.values())

    # ------------------------------------------------------------------ #
    # Scoring
    # ------------------------------------------------------------------ #
    def _as_matrix(self, contexts):
        """Aceita lista/array/DataFrame e devolve (n, n_features) na ordem do contrato."""
        if isinstance(contexts, pd.DataFrame):
            faltando = [c for c in self.feature_names if c not in contexts.columns]
            if faltando:
                raise ValueError(f"Features ausentes no contexto: {faltando}")
            X = contexts[self.feature_names].to_numpy(dtype=float)
        else:
            X = np.atleast_2d(np.asarray(contexts, dtype=float))
        if X.shape[1] != len(self.feature_names):
            raise ValueError(
                f"Esperava {len(self.feature_names)} features {self.feature_names}, "
                f"recebi {X.shape[1]}."
            )
        return X

    def predict_uplifts(self, contexts):
        """Uplift estimado de cada tratamento contra o controle. (n, n_treatments)"""
        X = self._as_matrix(contexts)
        out = np.zeros((X.shape[0], len(self.treatments)))
        for j, trt in enumerate(self.treatments):
            if self.is_trained[trt]:
                out[:, j] = self.models[trt].predict(X)
        return out

    def _greedy_index(self, X):
        """
        Índice (dentro de self.arms) do braço que a EXPLOTAÇÃO escolheria.

        A DECISÃO CAUSAL: se nenhuma mensagem gera aumento de chance, a melhor
        ação é NÃO MANDAR NADA. A margem evita agir com base em ruído do
        estimador (o máximo de vários CATEs ruidosos é otimista por construção
        — winner's curse).
        """
        idx_control = self.arms.index(self.control_name)
        uplifts = self.predict_uplifts(X)
        melhor = uplifts.argmax(axis=1)
        return np.where(uplifts.max(axis=1) > self.uplift_margin, melhor, idx_control)

    def greedy_action(self, contexts):
        """Ação da explotação pura, sem exploração. Para auditoria e simulação."""
        X = self._as_matrix(contexts)
        if self.in_warmup:
            return np.full(X.shape[0], self.control_name)
        return np.array(self.arms)[self._greedy_index(X)]

    def propensity_matrix(self, contexts):
        """
        Distribuição de probabilidade COMPLETA sobre os braços, uma linha por cliente.

        É a distribuição inteira que precisa ser logada (não só a do braço sorteado):
        para estimar o uplift do 'pix' contra o controle, o treino precisa saber qual
        era a chance do 'pix' até para os clientes que caíram no controle.
        """
        X = self._as_matrix(contexts)
        n, arms = X.shape[0], self.arms
        idx_control = arms.index(self.control_name)

        # ---- Warm-up: sem modelo, randomiza uniforme ----------------------
        # Cuidado: assumir uplift = 0 aqui NÃO força exploração — força o
        # controle, porque a regra gulosa manda calar quando nada é positivo.
        # Isso concentraria ~84% do tráfego no silêncio e o modelo nunca
        # aprenderia. No warm-up o controle também precisa de volume, senão
        # não há contrafactual para estimar uplift nenhum.
        if self.in_warmup:
            return pd.DataFrame(
                np.full((n, len(arms)), 1.0 / len(arms)),
                columns=self.prop_cols,
                index=getattr(contexts, "index", None) if isinstance(contexts, pd.DataFrame) else None,
            )

        # ---- Base: todo braço tem chance epsilon/n_arms na exploração ------
        probs = np.full((n, len(arms)), self.epsilon / len(arms))

        # ---- Piso do controle: ele é o contrafactual de todos os modelos ---
        probs[:, idx_control] += self.control_floor

        # ---- O braço guloso acumula a massa restante -----------------------
        probs[np.arange(n), self._greedy_index(X)] += 1 - self.epsilon - self.control_floor

        return pd.DataFrame(
            probs,
            columns=self.prop_cols,
            index=getattr(contexts, "index", None) if isinstance(contexts, pd.DataFrame) else None,
        )

    def recommend_batch(self, contexts):
        """Sorteia uma ação por linha. Devolve (ações, DataFrame de propensões)."""
        probs = self.propensity_matrix(contexts)
        p = probs.to_numpy()
        p = p / p.sum(axis=1, keepdims=True)  # blinda contra erro de ponto flutuante

        # Sorteio vetorizado direto da distribuição: garante que a propensão
        # logada é exatamente a probabilidade com que a ação foi escolhida.
        acumulado = p.cumsum(axis=1)
        u = self.rng.random((p.shape[0], 1))
        escolhido = (u > acumulado).sum(axis=1)
        acoes = np.array(self.arms)[escolhido]
        return acoes, probs

    def recommend(self, context_features):
        """Caminho de produção (um cliente): devolve (ação, dict de propensões)."""
        acoes, probs = self.recommend_batch(context_features)
        return acoes[0], probs.iloc[0].rename(lambda c: c[2:]).to_dict()

    # ------------------------------------------------------------------ #
    # Treino em batch
    # ------------------------------------------------------------------ #
    def train_batch(self, batch_data, verbose=True):
        """
        batch_data: log do que aconteceu após a janela de atribuição (ex: 7 dias).
        Colunas exigidas: self.feature_names + ['action', 'converted'] + self.prop_cols.
        Colunas extras (id do cliente, timestamp, safra) são ignoradas — o modelo só
        enxerga o que está em feature_names, na ordem do contrato.
        """
        obrigatorias = self.feature_names + ["action", "converted"] + self.prop_cols
        faltando = [c for c in obrigatorias if c not in batch_data.columns]
        if faltando:
            raise ValueError(f"Colunas ausentes no log: {faltando}")

        relatorio = {}

        for trt in self.treatments:
            # Só quem recebeu ESTE tratamento ou o CONTROLE entra no modelo dele.
            df = batch_data[batch_data["action"].isin([trt, self.control_name])]

            p_trt = df[f"p_{trt}"].to_numpy(dtype=float)
            p_ctrl = df[f"p_{self.control_name}"].to_numpy(dtype=float)
            denom = p_trt + p_ctrl

            # Linhas com propensão ausente/zerada não são identificáveis: descarta
            # ANTES de dividir. Clipar depois não adianta — np.clip(nan) == nan e o
            # XGBoost treinaria em alvo NaN.
            valido = np.isfinite(denom) & (denom > 0)
            df, p_trt, denom = df[valido], p_trt[valido], denom[valido]

            w = (df["action"].to_numpy() == trt).astype(float)
            y = df["converted"].to_numpy(dtype=float)

            n = len(df)
            n_trt, n_ctrl = int(w.sum()), int(n - w.sum())
            n_conv = int(y.sum())

            # ---------------------------------------------------------------
            # GUARDAS DE IDENTIFICAÇÃO
            # Três contagens, porque o volume total não protege contra nada por
            # si. As duas últimas falham em SILÊNCIO — sem elas o braço é
            # marcado como treinado e passa a decidir tráfego real:
            #
            #   n          : volume, o que o XGBoost precisa para formar folhas.
            #   por grupo  : 600 linhas com 3 tratadas dão um CATE em cima de 3
            #                observações — um número, com o sinal a esmo.
            #   conversões : Y=0 entra em Y* como zero EXATO, então quem carrega
            #                sinal é o convertido, não a linha. Com ZERO
            #                conversões o alvo é identicamente 0, o XGBoost
            #                ajusta a constante zero e is_trained fica True.
            #                Como 0 nunca passa de uplift_margin, esse braço não
            #                é escolhido nunca mais — e in_warmup fica False, ou
            #                seja, não há warm-up que o resgate. O braço morre
            #                parecendo saudável.
            #
            # Estas contagens são do SUBSET (tratado + controle), não do batch.
            # ---------------------------------------------------------------
            if n < self.min_samples:
                motivo = f"amostra insuficiente (n={n} < {self.min_samples})"
            elif min(n_trt, n_ctrl) < self.min_group_samples:
                motivo = (f"grupo pequeno (tratado={n_trt}, controle={n_ctrl}, "
                          f"mínimo={self.min_group_samples})")
            elif n_conv < self.min_conversions:
                motivo = f"conversões insuficientes ({n_conv} < {self.min_conversions})"
            else:
                motivo = None

            if motivo is not None:
                self.batches_since_fit[trt] += 1
                if self.batches_since_fit[trt] > self.max_batches_stale:
                    self.is_trained[trt] = False  # modelo obsoleto: volta pro warm-up
                relatorio[trt] = dict(
                    n=n, n_tratado=n_trt, n_controle=n_ctrl, n_convertido=n_conv,
                    treinado=False, motivo=motivo,
                )
                continue

            # ---------------------------------------------------------------
            # A PROPENSÃO CORRETA: p = P(W=1 | X, braço ∈ {trt, controle})
            # Não é a propensão da ação tomada. Como dentro do subset só existem
            # duas opções, renormalizamos no par:
            #     p = P(trt) / (P(trt) + P(controle))
            # Vale para TODA linha do subset, tratada ou controle — é isso que a
            # fórmula do Y* exige. Condicionar no par não enviesa porque a
            # atribuição é aleatória dado X.
            # ---------------------------------------------------------------
            p_bruto = p_trt / denom
            p = np.clip(p_bruto, self.propensity_floor, 1 - self.propensity_floor)
            taxa_clip = float(np.mean(p != p_bruto))

            # ---------------------------------------------------------------
            # O CORAÇÃO DO ALGORITMO: O RESULTADO TRANSFORMADO (Y*)
            # Transforma a conversão bruta em uplift incremental.
            #     Y* = Y * (W/p - (1-W)/(1-p))  ==  Y * (W - p) / (p(1-p))
            # ---------------------------------------------------------------
            y_star = y * (w / p - (1 - w) / (1 - p))

            X = df[self.feature_names].to_numpy(dtype=float)
            self.models[trt].fit(X, y_star)
            self.is_trained[trt] = True
            self.batches_since_fit[trt] = 0

            relatorio[trt] = dict(
                n=n,
                n_tratado=n_trt,
                n_controle=n_ctrl,
                n_convertido=n_conv,
                treinado=True,
                # ATE por IPW na distribuição do SUBSET, que não é a da população
                # quando a política já explota (o subset sobre-representa os
                # contextos em que trt ou controle é a ação gulosa). Serve para
                # sanidade do sinal, não como métrica de drift do efeito.
                uplift_medio=float(y_star.mean()),
                taxa_clip=taxa_clip,
            )

        if verbose:
            fase = "WARM-UP (randomização uniforme)" if self.in_warmup else "EXPLOTAÇÃO"
            print(f"Modelos atualizados. Fase atual: {fase}")
            for trt, r in relatorio.items():
                if r["treinado"]:
                    print(
                        f"  {trt:<14} n={r['n']:>6}  ATE_ipw={r['uplift_medio']:+.4f}"
                        f"  clip={r['taxa_clip']:.1%}"
                    )
                else:
                    print(f"  {trt:<14} n={r['n']:>6}  PULADO ({r['motivo']})")

        return relatorio

## 2. Um mundo sintético com a verdade conhecida

Rodar o código sem erro não prova nada: com `converted` aleatório, os modelos aprendem
ruído e o notebook imprime resultados iguaizinhos. O único teste que de fato valida a
implementação do $Y^*$ é comparar o CATE estimado contra um CATE **verdadeiro**, e para
isso é preciso um mundo simulado.

O simulador abaixo tem heterogeneidade real (cada assunto funciona para um segmento
diferente), um assunto que **sempre atrapalha** (`seguro`) e um segmento em que **todos**
os assuntos têm uplift negativo — ali o acerto é calar a boca.

In [ ]:
TREATMENTS = ["pix", "pagamento", "seguro", "investimento"]
FEATURES = ["idade", "renda", "ios"]
CONTROLE = "controle"


class SimuladorDeClientes:
    """
    Mundo sintético com CATE VERDADEIRO conhecido. Sem isso não dá para dizer se
    o bandit está certo: num log real o uplift nunca é observado.

    Regras da verdade (é isso que o modelo tem que redescobrir sozinho):
      pix          : +12 p.p. para quem tem menos de 30 anos, -1 p.p. no resto
      pagamento    : +5  p.p. para usuários de iOS,           -1 p.p. no resto
      seguro       : -3  p.p. sempre (mensagem que só atrapalha)
      investimento : +14 p.p. para renda > 8k,                -2 p.p. no resto

    Existe um segmento (30+, Android, renda baixa) em que TODOS os assuntos têm
    uplift negativo: ali a ação ótima é o silêncio.
    """

    def __init__(self, seed=42):
        self.rng = np.random.default_rng(seed)

    def gerar_contextos(self, n):
        return pd.DataFrame({
            "idade": self.rng.integers(18, 65, n),
            "renda": self.rng.uniform(1000, 15000, n),
            "ios": self.rng.integers(0, 2, n),
        })

    @staticmethod
    def taxa_base(df):
        """P(converter | controle) — a conversão que aconteceria sem mensagem."""
        z = (-1.6 + 0.02 * (df["idade"] - 40)
             + 0.00004 * (df["renda"] - 7000)
             + 0.25 * df["ios"])
        return 1 / (1 + np.exp(-z))

    @staticmethod
    def uplift_verdadeiro(df):
        """CATE real de cada tratamento contra o controle."""
        n = len(df)
        return pd.DataFrame({
            "pix": np.where(df["idade"] < 30, 0.12, -0.01),
            "pagamento": np.where(df["ios"] == 1, 0.05, -0.01),
            "seguro": np.full(n, -0.03),
            "investimento": np.where(df["renda"] > 8000, 0.14, -0.02),
        }, index=df.index)

    def acao_otima(self, df):
        """A política oracular: melhor assunto, ou silêncio se nada ajuda."""
        u = self.uplift_verdadeiro(df)
        melhor = u.idxmax(axis=1)
        return melhor.where(u.max(axis=1) > 0, CONTROLE)

    def taxa_esperada(self, df, acoes):
        """E[conversão] de uma política — sem ruído de amostragem."""
        u = self.uplift_verdadeiro(df)
        acoes = pd.Series(acoes, index=df.index)
        efeito = np.array([
            0.0 if a == CONTROLE else u.at[i, a]
            for i, a in acoes.items()
        ])
        return float((self.taxa_base(df) + efeito).mean())

    def simular_conversoes(self, df, acoes):
        """Joga a moeda: converteu ou não, dado o contexto e a ação recebida."""
        u = self.uplift_verdadeiro(df)
        acoes = pd.Series(acoes, index=df.index)
        efeito = np.array([
            0.0 if a == CONTROLE else u.at[i, a]
            for i, a in acoes.items()
        ])
        p = np.clip(self.taxa_base(df) + efeito, 0.001, 0.999)
        return (self.rng.random(len(df)) < p).astype(int)

## 3. Um ciclo completo: decidir → logar → esperar 7 dias → treinar

Repare nas colunas `cliente_id` e `safra` no log: elas existem de propósito, para
mostrar que o contrato de features as ignora. No código antigo — que montava `X` com
`df.drop(columns=log_cols)` — elas teriam virado features do modelo.

In [ ]:
sim = SimuladorDeClientes(seed=42)
bandit = CausalContextualBandit(
    treatments=TREATMENTS,
    feature_names=FEATURES,
    epsilon=0.20,
    control_floor=0.15,
    min_samples=500,
    random_state=7,
)

# --- Dia 1: o motor decide. Em warm-up, randomiza uniforme entre os 5 braços. --
N = 40_000
contextos = sim.gerar_contextos(N)
acoes, propensoes = bandit.recommend_batch(contextos)

# --- O que vai para o Kafka / Data Warehouse ---------------------------------
# Logamos a propensão de TODOS os braços, não só a do escolhido.
log = pd.concat([contextos, propensoes], axis=1)
log["cliente_id"] = np.arange(N)          # coluna extra de propósito:
log["safra"] = "2026-07"                  # o contrato de features a ignora
log["action"] = acoes

# --- ... 7 DIAS SE PASSAM ... o resultado volta ------------------------------
log["converted"] = sim.simular_conversoes(contextos, acoes)

print(f"Log: {len(log)} linhas | conversão bruta: {log['converted'].mean():.3f}")
print(log["action"].value_counts().to_string(), "\n")

# --- Job diário: atualiza os modelos de uplift -------------------------------
bandit.train_batch(log)

### Validação 1 — o CATE estimado bate com o verdadeiro?

O que importa não é acertar a média (o ATE), e sim a **heterogeneidade**: identificar
que `pix` funciona para jovem, `investimento` para renda alta, e que `seguro` nunca
funciona. É a heterogeneidade que justifica ter um bandit contextual em vez de um
teste A/B com o vencedor único.

In [ ]:
# =============================================================================
# VALIDAÇÃO 1: o modelo recupera o CATE verdadeiro?
# Num log real isso é impossível de medir. Aqui é possível, e é o único teste
# que de fato prova que a implementação do Y* está correta.
# =============================================================================
teste = sim.gerar_contextos(20_000)
estimado = pd.DataFrame(bandit.predict_uplifts(teste), columns=TREATMENTS, index=teste.index)
real = sim.uplift_verdadeiro(teste)

print("Uplift médio (ATE) — verdadeiro vs estimado")
print(f"{'assunto':<14}{'real':>9}{'estimado':>11}{'corr':>8}{'MAE':>9}")
for trt in TREATMENTS:
    # 'seguro' tem uplift constante: correlação é indefinida, não é sinal de erro.
    corr = ("  const" if real[trt].std() == 0
            else f"{np.corrcoef(estimado[trt], real[trt])[0, 1]:>6.3f}")
    mae = np.abs(estimado[trt] - real[trt]).mean()
    print(f"{trt:<14}{real[trt].mean():>+9.4f}{estimado[trt].mean():>+11.4f}{corr:>8}{mae:>9.4f}")

# --- O que importa de verdade: acertar a HETEROGENEIDADE por segmento --------
segmentos = {
    "idade < 30":  teste["idade"] < 30,
    "idade >= 30": teste["idade"] >= 30,
    "iOS":         teste["ios"] == 1,
    "Android":     teste["ios"] == 0,
    "renda > 8k":  teste["renda"] > 8000,
    "renda <= 8k": teste["renda"] <= 8000,
}
tabela = pd.DataFrame({
    f"{trt}_{tipo}": {nome: fonte.loc[m, trt].mean() for nome, m in segmentos.items()}
    for trt in TREATMENTS
    for tipo, fonte in (("real", real), ("est", estimado))
})
print("\nUplift por segmento (real vs estimado)")
print(tabela.round(3).to_string())

### Validação 2 — a política decide bem?

Acertar o CATE é meio; decidir bem é o fim. Um modelo pode ter erro alto e ainda assim
escolher o braço certo, e vice-versa. A régua aqui é quanto do ganho máximo possível
sobre "nunca mandar nada" a política captura — com o oráculo em 100% e o silêncio em 0%.

Compare também com as políticas ingênuas: mandar sempre o melhor assunto único é a
alternativa realista que um teste A/B tradicional entregaria.

In [ ]:
# =============================================================================
# VALIDAÇÃO 2: a POLÍTICA é boa? (acertar o CATE é meio; decidir bem é o fim)
# =============================================================================
otima = sim.acao_otima(teste)
politica = pd.Series(bandit.greedy_action(teste), index=teste.index)

acerto = (politica == otima).mean()
mask_silencio = otima == CONTROLE
print(f"Concordância com a política ótima: {acerto:.1%}")
print(f"Clientes em que calar é o ótimo:   {mask_silencio.mean():.1%} "
      f"(o motor cala em {(politica[mask_silencio] == CONTROLE).mean():.1%} deles)")

print("\nMatriz de decisão (linhas = ótimo, colunas = escolhido)")
print(pd.crosstab(otima, politica).to_string())

# --- Conversão esperada de cada política -------------------------------------
alternativas = {
    "Oráculo (teto)":        otima,
    "Bandit causal":         politica,
    "Sempre controle":       pd.Series(CONTROLE, index=teste.index),
    "Assunto aleatório":     pd.Series(sim.rng.choice(TREATMENTS, len(teste)), index=teste.index),
}
for trt in TREATMENTS:
    alternativas[f"Sempre '{trt}'"] = pd.Series(trt, index=teste.index)

teto = sim.taxa_esperada(teste, otima)
piso = sim.taxa_esperada(teste, pd.Series(CONTROLE, index=teste.index))

print(f"\n{'política':<22}{'conversão':>11}{'vs controle':>13}{'% do ganho máx':>17}")
for nome, acoes_pol in alternativas.items():
    taxa = sim.taxa_esperada(teste, acoes_pol)
    print(f"{nome:<22}{taxa:>11.4f}{taxa - piso:>+13.4f}{(taxa - piso) / (teto - piso):>16.0%}")

# --- A alavanca da margem ----------------------------------------------------
# Perto de zero o CATE estimado é dominado por ruído, e o máximo de 4 estimativas
# ruidosas é otimista por construção (winner's curse): o motor manda mensagem
# achando que ganha +0.001 quando na verdade perde. Exigir uma margem mínima
# compra silêncio nos casos duvidosos.
print("\nEfeito da margem de uplift")
print(f"{'margem':>8}{'silêncio':>11}{'acerto':>9}{'% ganho':>9}")
margem_original = bandit.uplift_margin
for m in [0.0, 0.005, 0.01, 0.02, 0.03]:
    bandit.uplift_margin = m
    pol = pd.Series(bandit.greedy_action(teste), index=teste.index)
    taxa = sim.taxa_esperada(teste, pol)
    print(f"{m:>8.3f}{(pol == CONTROLE).mean():>11.1%}{(pol == otima).mean():>9.1%}"
          f"{(taxa - piso) / (teto - piso):>9.0%}")
bandit.uplift_margin = margem_original

### Validação 3 — o ciclo fechado se sustenta?

Este é o teste que separa um bandit que funciona de um que parece funcionar. A partir
da segunda rodada o motor consome o log que ele mesmo gerou, com propensões que já não
são uniformes. Se a renormalização da propensão estivesse errada, o viés se realimentaria
e a coluna `aprendido` cairia rodada após rodada em vez de subir.

Duas métricas separadas de propósito:

| coluna | o que mede |
|---|---|
| `realizado` | conversão da política que rodou de fato — **já paga** o custo da exploração e do piso de controle. É o que o negócio vê hoje. |
| `aprendido` | conversão da política gulosa se você desligasse a exploração agora. É o que o modelo de fato aprendeu. |

O intervalo entre as duas é o preço da exploração. Ele não é desperdício: é o que faz
`aprendido` subir. Rode com `control_floor=0.0` para ver `realizado` subir alguns pontos
no começo e `aprendido` estacionar bem mais baixo.

In [ ]:
# =============================================================================
# VALIDAÇÃO 3: o ciclo fechado. Cada rodada = um dia de tráfego + o job de treino.
# A partir da rodada 2 o motor já explota, então passa a gerar o PRÓPRIO log — e
# as propensões deixam de ser uniformes. É aqui que um erro na renormalização da
# propensão apareceria: a política degeneraria ao longo do tempo em vez de subir.
#
# Duas métricas, porque medem coisas diferentes:
#   realizado = conversão da política que rodou de fato (JÁ PAGA o custo da
#               exploração e do piso de controle) -> é o que o negócio vê hoje.
#   aprendido = conversão da política gulosa, se você desligasse a exploração
#               agora -> é o que o modelo de fato aprendeu.
# Ambas em % do ganho máximo possível sobre "nunca mandar nada".
# =============================================================================
bandit_online = CausalContextualBandit(
    treatments=TREATMENTS, feature_names=FEATURES,
    epsilon=0.20, control_floor=0.15, min_samples=500, random_state=99,
)
sim_online = SimuladorDeClientes(seed=2024)

N_RODADA, N_RODADAS = 10_000, 10
historico, acumulado = [], []

for rodada in range(1, N_RODADAS + 1):
    ctx = sim_online.gerar_contextos(N_RODADA)
    acoes_r, props_r = bandit_online.recommend_batch(ctx)

    lote = pd.concat([ctx, props_r], axis=1)
    lote["action"] = acoes_r
    lote["converted"] = sim_online.simular_conversoes(ctx, acoes_r)

    # Métricas ANTES do treino: refletem o modelo que tomou as decisões da rodada.
    otima_r = sim_online.acao_otima(ctx)
    teto = sim_online.taxa_esperada(ctx, otima_r)
    piso = sim_online.taxa_esperada(ctx, pd.Series(CONTROLE, index=ctx.index))
    greedy_r = pd.Series(bandit_online.greedy_action(ctx), index=ctx.index)
    ganho = lambda acoes: (sim_online.taxa_esperada(ctx, acoes) - piso) / (teto - piso)

    historico.append(dict(
        rodada=rodada,
        fase="warm-up" if bandit_online.in_warmup else "explota",
        conv_obs=lote["converted"].mean(),
        realizado=ganho(acoes_r),
        aprendido=ganho(greedy_r),
        acerto=(greedy_r == otima_r).mean(),
        pct_silencio=(pd.Series(acoes_r) == CONTROLE).mean(),
    ))

    # O treino usa a janela acumulada (em produção: últimos K dias do DW).
    acumulado.append(lote)
    bandit_online.train_batch(pd.concat(acumulado, ignore_index=True), verbose=False)

print(pd.DataFrame(historico).to_string(
    index=False,
    formatters={
        "conv_obs": "{:.4f}".format, "realizado": "{:.0%}".format,
        "aprendido": "{:.0%}".format, "acerto": "{:.1%}".format,
        "pct_silencio": "{:.1%}".format,
    },
))

## 4. O caminho de produção

Um cliente por vez, no momento em que a conta é aberta.

In [ ]:
# =============================================================================
# O CAMINHO DE PRODUÇÃO (um cliente por vez, na hora que a conta é aberta)
# =============================================================================
perfis = {
    "jovem, renda baixa, Android": dict(idade=24, renda=2500.0, ios=0),
    "45 anos, renda alta, iOS":    dict(idade=45, renda=12000.0, ios=1),
    "50 anos, renda baixa, Android": dict(idade=50, renda=2200.0, ios=0),
}

for nome, perfil in perfis.items():
    # Passe um dict/DataFrame, não uma lista posicional: a ordem das features
    # deixa de ser um acordo tácito e passa a ser validada.
    ctx = pd.DataFrame([perfil])
    assunto, props = bandit_online.recommend(ctx)
    uplifts = dict(zip(TREATMENTS, bandit_online.predict_uplifts(ctx)[0]))

    print(f"\n{nome}")
    print("  uplift estimado: " + "  ".join(f"{k}={v:+.3f}" for k, v in uplifts.items()))
    print(f"  guloso: {bandit_online.greedy_action(ctx)[0]}  ->  sorteado: {assunto} "
          f"(p={props[assunto]:.2f})")
    print(f"  ótimo (oráculo): {sim_online.acao_otima(ctx).iloc[0]}")

# Se 'assunto' != controle, é ele que vai para o LLM gerar a mensagem.
# Se for controle, não se manda nada — e a linha É logada mesmo assim, porque é
# ela que serve de contrafactual no treino de amanhã.

# Repare no terceiro perfil: o ótimo é calar, mas o motor vê um uplift de +0.001 e
# manda mensagem. É o winner's curse em ação, e é exatamente o caso que a margem
# resolve — com uplift_margin=0.01 esse cliente entraria no silêncio.
bandit_online.uplift_margin = 0.01
print("\nMesmo terceiro perfil, agora com uplift_margin=0.01: "
      f"{bandit_online.greedy_action(pd.DataFrame([perfis['50 anos, renda baixa, Android']]))[0]}")
bandit_online.uplift_margin = 0.0

## O que ainda falta antes de ir para produção

O que este notebook resolve termina no algoritmo. Os itens abaixo são de engenharia e
de processo, e cada um deles já derrubou um bandit em produção:

**Integridade do log**
- A linha do cliente que caiu no **controle** precisa ser gravada. É contraintuitivo
  (não houve envio, não há evento), mas sem ela não existe contrafactual e o modelo
  inteiro para de ser identificável.
- As features têm que ser as **do momento da decisão**, não as de hoje. Ler o cadastro
  atual no job de treino vaza futuro: renda e engajamento mudaram *por causa* da
  mensagem. Isso pede um feature store com viagem no tempo ou um snapshot no log.
- Propensão gravada junto com a decisão, no mesmo commit. Recalcular depois a partir do
  modelo "que estava no ar" é uma fonte clássica de erro silencioso.

**Ciclo de vida do modelo**
- Versione o modelo no log (`model_version`) — sem isso não dá para explicar uma queda.
- `max_batches_stale` derruba o braço para warm-up quando ele fica sem dados; ligue isso
  a um alerta, porque quase sempre significa que o tráfego secou por algum bug a montante.
- Janela de treino: aqui os batches são acumulados indefinidamente. Em produção o efeito
  muda com sazonalidade e com a própria mensagem — use janela deslizante e monitore se
  o ATE por braço se move.

**Decisão de negócio**
- `uplift_margin` é o dial entre agir e calar; a célula de validação 2 mostra o efeito.
  Se mandar mensagem tem custo (SMS, risco de descadastro), a margem deveria ser o custo
  convertido em pontos de conversão, não zero.
- Regras de negócio (frequência máxima, opt-out, clientes elegíveis) entram **antes** do
  bandit, filtrando quem chega até ele — não como pós-filtro da ação escolhida, senão a
  propensão logada deixa de bater com a ação executada e o treino quebra.

**Estatística**
- Esta implementação usa o transformed outcome puro. Ele é não-enviesado mas ruidoso.
  Se o volume for apertado, um **X-learner** ou **DR-learner** (`econml`, `causalml`)
  entrega o mesmo CATE com bem menos variância — a interface de decisão desta classe não
  muda, só o que roda dentro de `train_batch`.
- Antes de confiar no ganho, valide off-policy no log real (IPS / SNIPS / doubly robust)
  e confirme com um holdout aleatório permanente. A simulação acima prova que o
  *algoritmo* funciona, não que ele vai funcionar nos *seus* dados.